# 0. Start

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from copy import deepcopy
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score

In [2]:
from catboost import CatBoostClassifier, CatBoostRegressor

classification = CatBoostClassifier(
    iterations = 300,
    learning_rate=0.05,
    max_depth = 7,
    auto_class_weights='Balanced',
    verbose = 100,
    random_state = 67
)

regression = CatBoostRegressor(
    iterations = 300,
    learning_rate=0.05,
    max_depth = 8,
    verbose = 100,
    random_state = 67
)

In [3]:
def adversarial(ts, tr, c, ct, 
                return_model = False):

    columns = c + ct
    X = pd.concat([tr[columns], ts[columns]]).reset_index(drop = True)
    y = np.zeros(len(tr)).tolist() + np.ones(len(ts)).tolist()
    m = deepcopy(classification)
    m.set_params(cat_features = ct)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.2, stratify = y, random_state = 67)
    
    m.fit(X_train, y_train)
    pred = m.predict_proba(X_test)[:, 1]
    metric = roc_auc_score(y_test, pred)

    if (return_model):
        return m, metric
    return metric

In [4]:
def imp(m, t):
    return pd.DataFrame({
        'imp':m.feature_importances_,
        'col':t
    }).sort_values('imp', ascending=False)

In [5]:
def crossval(X, y, model_ref, metric, spl = 5, return_model = False, stratify = True):
    hist = []

    if (stratify):
        dt = StratifiedKFold(n_splits = spl, shuffle=True, random_state=67).split(X, y)
    else:
        dt = KFold(n_splits = spl, shuffle=True, random_state=67).split(X)

    for tr_ids, ts_ids in dt:
        X_train, y_train, X_test, y_test = X.loc[tr_ids], \
            y.loc[tr_ids], X.loc[ts_ids], y.loc[ts_ids]
        model = deepcopy(model_ref)

        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        val = metric(y_test, pred)

        print(round(val, 4))
        hist.append(val)

    hist = np.array(hist)
    if (return_model):
        return model, hist
    return hist

# 1. Clients EDA

### 1.1 Adversarial

In [6]:
c_train = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\clients_train.csv')
c_test = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\clients_test.csv')

display(c_train)
display(c_test)

,client_id,branch_id,snapshot_date,account_age_days,avg_inflow_30d,avg_inflow_90d,avg_outflow_30d,avg_outflow_90d,balance_mean_30d,balance_std_30d,txn_count_30d,spend_food_share,spend_transport_share,spend_entertainment_share,has_mobile_app,region_code,inflow_next_30d
0,C000000,B008,2026-01-31,3464,757.91,678.30,534.19,545.17,210.42,212.09,1,0.40334,0.22646,0.37021,0,3,203.18
1,C000001,B005,2026-01-31,721,3101.35,2553.16,1531.37,1753.46,541.20,676.61,2,0.25591,0.23956,0.50453,1,0,3096.53
2,C000002,B005,2026-01-31,925,2423.63,2741.22,1389.53,2072.41,2059.97,563.28,5,0.41003,0.45392,0.13605,1,0,2109.75
3,C000003,B000,2026-01-31,3338,4289.27,4329.95,1771.59,2028.21,2729.84,634.24,8,0.62746,0.34780,0.02474,1,0,3833.00
4,C000004,B008,2026-01-31,1052,2365.24,2157.34,2125.83,1983.13,2024.76,366.04,9,0.33661,0.63875,0.02464,0,3,1760.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,C000395,B002,2026-01-31,1332,1491.52,1880.84,1406.53,1629.67,1226.03,5.00,5,0.45081,0.17557,0.37363,1,2,1214.25
396,C000396,B008,2026-01-31,829,1436.56,1234.66,733.84,795.49,1653.41,358.45,1,0.69698,0.19596,0.10706,1,3,1693.23
397,C000397,B002,2026-01-31,2922,4774.60,3523.36,2105.38,2104.64,2449.32,1011.87,5,0.60050,0.20060,0.19890,1,2,3598.35
398,C000398,B011,2026-01-31,2194,5122.93,5266.66,5386.49,4727.09,314.17,837.63,14,0.62722,0.15959,0.21319,0,1,4856.51


,client_id,branch_id,snapshot_date,account_age_days,avg_inflow_30d,avg_inflow_90d,avg_outflow_30d,avg_outflow_90d,balance_mean_30d,balance_std_30d,txn_count_30d,spend_food_share,spend_transport_share,spend_entertainment_share,has_mobile_app,region_code
0,T000000,N004,2026-01-31,2928,844.69,1044.88,596.33,497.98,2244.69,196.86,1,0.26090,0.55782,0.18128,0,4
1,T000001,N004,2026-01-31,1133,1524.44,1432.47,869.09,781.97,1213.74,67.36,4,0.49366,0.31342,0.19292,1,4
2,T000002,N003,2026-01-31,719,3931.69,4401.03,3359.37,3547.70,1910.50,861.31,14,0.22556,0.66103,0.11341,0,3
3,T000003,N004,2026-01-31,601,1299.35,1169.28,663.47,810.35,935.84,305.19,1,0.27502,0.39029,0.33469,0,4
4,T000004,N001,2026-01-31,2766,1894.47,2230.83,1662.40,1515.00,1115.33,603.70,8,0.44290,0.37334,0.18377,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,T000155,B000,2026-01-31,2687,4636.27,4465.88,2551.76,2642.48,7093.40,731.44,14,0.37234,0.53622,0.09144,1,0
156,T000156,N001,2026-01-31,3408,2643.56,2382.29,1159.43,1836.58,1809.36,24.18,5,0.59378,0.27341,0.13281,0,1
157,T000157,B000,2026-01-31,1099,1571.31,1418.35,924.39,939.66,1121.93,383.34,4,0.51361,0.27694,0.20944,1,0
158,T000158,B001,2026-01-31,202,3340.45,2799.48,2924.32,2340.86,1752.35,456.13,15,0.22982,0.27608,0.49410,0,1


In [7]:
num = ['account_age_days', 'avg_inflow_30d', 'avg_inflow_90d', 'avg_outflow_30d', 'avg_outflow_90d', 'balance_mean_30d', 
        'balance_std_30d', 'txn_count_30d', 'spend_food_share', 'spend_transport_share', 'spend_entertainment_share']
cat = ['has_mobile_app', 'region_code']
cols = cat + num

m, h = adversarial(c_train, c_test, num, cat, return_model=True)
print(h)
imp(m, cols)

0:	learn: 0.6778699	total: 189ms	remaining: 56.5s
100:	learn: 0.3047312	total: 2.82s	remaining: 5.56s
200:	learn: 0.1409557	total: 5.33s	remaining: 2.63s
299:	learn: 0.0756279	total: 7.73s	remaining: 0us
0.47148437499999996


,imp,col
12,12.694423,spend_entertainment_share
0,10.449715,has_mobile_app
10,10.080949,spend_food_share
6,9.166784,avg_outflow_90d
8,7.889451,balance_std_30d
5,7.626760,avg_outflow_30d
9,7.481079,txn_count_30d
3,7.120367,avg_inflow_30d
1,6.595398,region_code
4,5.948082,avg_inflow_90d


### 1.2 Branch id

In [8]:
display(c_train.branch_id.value_counts())
display(c_test.branch_id.value_counts())

intersection = set(c_test.branch_id.unique()) & set(c_train.branch_id.unique())
print(intersection)

branch_id
B008    48
B010    42
B007    41
B005    40
B006    32
B002    31
B009    30
B000    29
B001    29
B004    28
B011    27
B003    23
Name: count, dtype: int64

branch_id
B001    33
N001    30
N004    22
N000    20
N003    19
N002    19
B000    17
Name: count, dtype: int64

{'B001', 'B000'}


In [9]:
from sklearn.metrics import accuracy_score
XN = pd.concat([c_train[cols].drop(columns = ['region_code']),
    c_test[cols].drop(columns = ['region_code'])], axis = 0).reset_index(drop = True)
yN = pd.concat([c_train['branch_id'], c_test['branch_id']], 
    axis = 0).reset_index(drop = True)

classification.set_params(verbose = 0)
classification.set_params(cat_features = ['has_mobile_app'])
h = crossval(XN, yN, classification, accuracy_score, spl = 3)
print(h.mean())

0.0695
0.0963
0.0914
0.08572441684396144


In [10]:
for i in c_train['branch_id'].unique():
    print(c_train[c_train['branch_id'] == i]['region_code'].unique())

[3]
[0]
[0]
[3]
[4]
[1]
[1]
[1]
[4]
[2]
[0]
[2]


### 1.3 Other

In [11]:
display(c_train.snapshot_date.value_counts())
display(c_test.snapshot_date.value_counts())
print()
display(c_train.has_mobile_app.value_counts())
display(c_test.has_mobile_app.value_counts())
print()
display(c_train.region_code.value_counts())
display(c_test.region_code.value_counts())

snapshot_date
2026-01-31    400
Name: count, dtype: int64

snapshot_date
2026-01-31    160
Name: count, dtype: int64

has_mobile_app
1    232
0    168
Name: count, dtype: int64

has_mobile_app
0    84
1    76
Name: count, dtype: int64

region_code
0    111
1     88
2     72
3     71
4     58
Name: count, dtype: int64

region_code
1    63
0    37
4    22
3    19
2    19
Name: count, dtype: int64

In [12]:
print((c_train['spend_food_share'] + c_train['spend_transport_share'] + 
    c_train['spend_entertainment_share']).mean())

0.999999948979592


### 1.4 NaN

In [13]:
display(c_train.isna().sum())
display(c_test.isna().sum())

client_id                     0
branch_id                     0
snapshot_date                 0
account_age_days              0
avg_inflow_30d                0
avg_inflow_90d                0
avg_outflow_30d               0
avg_outflow_90d               0
balance_mean_30d              0
balance_std_30d              14
txn_count_30d                 0
spend_food_share              0
spend_transport_share         0
spend_entertainment_share     8
has_mobile_app                0
region_code                   0
inflow_next_30d               0
dtype: int64

client_id                    0
branch_id                    0
snapshot_date                0
account_age_days             0
avg_inflow_30d               0
avg_inflow_90d               0
avg_outflow_30d              0
avg_outflow_90d              0
balance_mean_30d             0
balance_std_30d              3
txn_count_30d                0
spend_food_share             0
spend_transport_share        0
spend_entertainment_share    4
has_mobile_app               0
region_code                  0
dtype: int64

In [14]:
from sklearn.metrics import root_mean_squared_error

df = c_train[~c_train['balance_std_30d'].isna()].reset_index(drop = True)
X = df[cols].drop(columns = ['balance_std_30d'])
y = df['balance_std_30d']

regression.set_params(verbose = 0)
m, h = crossval(X, y, regression, root_mean_squared_error, return_model=True, stratify=False)
print(h.mean() / y.mean())

178.7689
202.2623
164.3988
188.7262
207.9695
0.4974588454171858


# 2. Clients Model

### 2.1 EDA usage

In [15]:
drop = ['snapshot_date', 'spend_entertainment_share']
c_train.drop(columns = drop, inplace = True)
c_test.drop(columns = drop, inplace = True)

med = c_train['balance_std_30d'].copy().median()
c_train['balance_std_30d'].fillna(med, inplace = True)
c_test['balance_std_30d'].fillna(med, inplace = True)
c_train

,client_id,branch_id,account_age_days,avg_inflow_30d,avg_inflow_90d,avg_outflow_30d,avg_outflow_90d,balance_mean_30d,balance_std_30d,txn_count_30d,spend_food_share,spend_transport_share,has_mobile_app,region_code,inflow_next_30d
0,C000000,B008,3464,757.91,678.30,534.19,545.17,210.42,212.09,1,0.40334,0.22646,0,3,203.18
1,C000001,B005,721,3101.35,2553.16,1531.37,1753.46,541.20,676.61,2,0.25591,0.23956,1,0,3096.53
2,C000002,B005,925,2423.63,2741.22,1389.53,2072.41,2059.97,563.28,5,0.41003,0.45392,1,0,2109.75
3,C000003,B000,3338,4289.27,4329.95,1771.59,2028.21,2729.84,634.24,8,0.62746,0.34780,1,0,3833.00
4,C000004,B008,1052,2365.24,2157.34,2125.83,1983.13,2024.76,366.04,9,0.33661,0.63875,0,3,1760.38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,C000395,B002,1332,1491.52,1880.84,1406.53,1629.67,1226.03,5.00,5,0.45081,0.17557,1,2,1214.25
396,C000396,B008,829,1436.56,1234.66,733.84,795.49,1653.41,358.45,1,0.69698,0.19596,1,3,1693.23
397,C000397,B002,2922,4774.60,3523.36,2105.38,2104.64,2449.32,1011.87,5,0.60050,0.20060,1,2,3598.35
398,C000398,B011,2194,5122.93,5266.66,5386.49,4727.09,314.17,837.63,14,0.62722,0.15959,0,1,4856.51


In [16]:
cols = [
    'has_mobile_app',
    'account_age_days',
    'avg_inflow_30d',
    'avg_inflow_90d',
    'avg_outflow_30d',
    'avg_outflow_90d',
    'balance_mean_30d',
    'balance_std_30d',
    'txn_count_30d',
    'spend_food_share',
    'spend_transport_share'
]

### 2.2 Feature selection

In [18]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import HuberRegressor

groups = c_train['branch_id']
X_r = c_train[cols]
y_r = c_train['inflow_next_30d']
gkf = GroupKFold(n_splits=5)
scores_random, scores_group = [], []

for tr, val in KFold(5, shuffle=True, random_state=67).split(X_r):
    m = HuberRegressor(alpha=10.0, max_iter=500).fit(X_r.iloc[tr], y_r.iloc[tr])
    scores_random.append(root_mean_squared_error(y_r.iloc[val], m.predict(X_r.iloc[val])))

for tr, val in gkf.split(X_r, y_r, groups):
    m = HuberRegressor(alpha=10.0, max_iter=500).fit(X_r.iloc[tr], y_r.iloc[tr])
    scores_group.append(root_mean_squared_error(y_r.iloc[val], m.predict(X_r.iloc[val])))

print(f"Random KFold: {np.mean(scores_random)}")
print(f"GroupKFold:   {np.mean(scores_group)}")

Random KFold: 456.13655206743596
GroupKFold:   457.8099128000058


In [19]:
from sklearn.feature_selection import RFECV
from sklearn.metrics import make_scorer

X = c_train[cols]
y = c_train['inflow_next_30d']

estimator = HuberRegressor(alpha=10.0, max_iter=500)
cv = KFold(n_splits=5, shuffle=True, random_state=67)
scorer = make_scorer(root_mean_squared_error, greater_is_better=False)

rfecv = RFECV(
    estimator=estimator, cv=cv, scoring=scorer,
    step=1, min_features_to_select=1, n_jobs=-1
)

rfecv.fit(X, y)

,estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance either through a ``coef_``attribute or through a ``feature_importances_`` attribute.,HuberRegresso... max_iter=500)
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.Note that the last iteration may remove fewer than ``step`` features inorder to reach ``min_features_to_select``.",1
,"min_features_to_select min_features_to_select: int, default=1The minimum number of features to be selected. This number of featureswill always be scored, even if the difference between the originalfeature count and ``min_features_to_select`` isn't divisible by``step``... versionadded:: 0.20",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If theestimator is not a classifier or if ``y`` is neither binary nor multiclass,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value of None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"scoring scoring: str or callable, default=NoneScoring method to evaluate the :class:`RFE` selectors' performance. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: the `estimator`'s :ref:`default evaluation criterion ` is used.",make_scorer(r...hod='predict')
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"n_jobs n_jobs: int or None, default=NoneNumber of cores to run in parallel while fitting across folds.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",-1
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance.For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case of:class:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
,"epsilon epsilon: float, default=1.35The parameter epsilon controls the number of samples that should beclassified as outliers. The smaller the epsilon, the more robust it isto outliers. Epsilon must be in the range `[1, inf)`.",1.35
,"max_iter max_iter: int, default=100Maximum number of iterations that``scipy.optimize.minimize(method=""L-BFGS-B"")`` should run for.",500
,"alpha alpha: float, default=0.0001Strength of the squared L2 regularization. Note that the penalty isequal to ``alpha * ||w||^2``.Must be in the range `[0, inf)`.",10.0


In [21]:
print(f"Cnt {rfecv.n_features_}")
print(f"RMSE {-rfecv.cv_results_['mean_test_score'].max()}\n")

res = pd.DataFrame({
    'col': X.columns,
    'rank': rfecv.ranking_,
    'select': rfecv.support_
}).sort_values('rank', ascending = False)

display(res)

#plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), 
#         -rfecv.cv_results_['mean_test_score'], marker='o')
#plt.show()

Cnt 4
RMSE 450.8085107941417



,col,rank,select
1,account_age_days,8,False
6,balance_mean_30d,7,False
5,avg_outflow_90d,6,False
4,avg_outflow_30d,5,False
10,spend_transport_share,4,False
7,balance_std_30d,3,False
3,avg_inflow_90d,2,False
2,avg_inflow_30d,1,True
0,has_mobile_app,1,True
8,txn_count_30d,1,True


### 2.3 Clients final config

In [22]:
new_cols = res[res['select']]['col'].tolist()
model = HuberRegressor(alpha=10.0, max_iter=500)
m, h = crossval(X_r[new_cols], y_r, model, root_mean_squared_error, stratify=False, return_model=True)

im = pd.DataFrame({
    "imp":m.coef_ * X[new_cols].mean().tolist(), 
    "col":new_cols
}).sort_values('imp', ascending = False).reset_index(drop = True)

print(h.mean())
im

493.1938
520.1483
465.2949
449.9177
514.928
488.6965403493001


,imp,col
0,1973.979816,avg_inflow_30d
1,42.225268,txn_count_30d
2,0.410684,has_mobile_app
3,-0.110809,spend_food_share


In [23]:
clients_cols = [
    'avg_inflow_30d',
    'has_mobile_app',
    'txn_count_30d',
    'spend_food_share'
]

clients_model = HuberRegressor(alpha=10.0, max_iter=500)

# 3. Credit EDA

### 3.1 Adversarial

In [24]:
r_train = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\credit_train.csv')
r_test = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\credit_test.csv')

display(r_train)
display(r_test)

,application_id,client_id,loan_amount,loan_term_months,monthly_payment,past_due_count_12m,credit_history_months,debt_to_income_known,overdue_next_60d
0,A000000,C000136,11023.19,36,336.96,0,78,0.07517,0
1,A000001,C000352,10283.72,6,1853.45,0,56,2.22623,1
2,A000002,C000282,7356.07,18,433.96,1,83,0.42699,0
3,A000003,C000116,7327.88,12,693.66,0,41,0.35629,1
4,A000004,C000280,12849.42,6,2476.53,0,90,1.80816,0
...,...,...,...,...,...,...,...,...,...
275,A000275,C000137,13907.93,6,2553.05,0,81,1.24564,0
276,A000276,C000164,3470.06,12,304.07,0,74,0.10363,0
277,A000277,C000313,17807.51,18,1160.55,0,46,0.36312,0
278,A000278,C000187,7906.24,18,488.57,0,22,0.21731,0


,application_id,client_id,loan_amount,loan_term_months,monthly_payment,past_due_count_12m,credit_history_months,debt_to_income_known
0,Z000000,T000010,7828.42,6,1440.38,0,52,0.75399
1,Z000001,T000057,9869.64,6,1856.26,0,41,0.99080
2,Z000002,T000042,8579.79,24,401.11,0,45,0.35393
3,Z000003,T000107,5415.85,12,524.70,0,39,0.21677
4,Z000004,T000024,10777.67,24,505.14,0,88,0.29136
...,...,...,...,...,...,...,...,...
99,Z000099,T000017,8323.50,6,1514.88,2,84,0.51510
100,Z000100,T000096,8045.12,12,718.51,1,29,0.60847
101,Z000101,T000068,6516.26,48,141.76,0,86,0.07122
102,Z000102,T000082,11854.49,24,514.12,1,80,0.39499


In [25]:
num = ['loan_amount', 'monthly_payment', 'credit_history_months', 
       'debt_to_income_known', 'loan_term_months', 'past_due_count_12m']
cat = []
trg = 'overdue_next_60d'
cols = cat + num

classification.set_params(verbose = 100)
m, h = adversarial(r_train, r_test, num, cat, return_model = True)
print(h)

0:	learn: 0.6855423	total: 9.82ms	remaining: 2.94s
100:	learn: 0.3573729	total: 496ms	remaining: 977ms
200:	learn: 0.1902022	total: 1.03s	remaining: 506ms
299:	learn: 0.1104723	total: 1.49s	remaining: 0us
0.471938775510204


### 3.2 Columns

In [26]:
display(r_train[trg].value_counts())
print()

for i in ['loan_term_months', 'past_due_count_12m']:
    display(r_train[i].value_counts())
    display(r_test[i].value_counts())
    print()

overdue_next_60d
0    233
1     47
Name: count, dtype: int64

loan_term_months
12    51
18    51
24    49
6     45
48    45
36    39
Name: count, dtype: int64

loan_term_months
48    21
24    19
36    18
6     16
18    16
12    14
Name: count, dtype: int64

past_due_count_12m
0    223
1     48
2      9
Name: count, dtype: int64

past_due_count_12m
0    79
1    21
2     4
Name: count, dtype: int64

In [27]:
print(r_train.isna().any().sum())
print(r_test.isna().any().sum())

0
0


In [28]:
a = set(c_train.client_id.unique())
b = set(r_train.client_id.unique())
print(len(a), len(b))
print(b.issubset(a))

400 280
True


### 3.3 Pure / Illegal prediction

In [29]:
def roc_auc_crossval(X, y, model_ref, spl = 5, return_model = False):
    hist = []

    for tr_ids, ts_ids in StratifiedKFold(n_splits = spl).split(X, y):
        X_train, y_train, X_test, y_test = X.loc[tr_ids], \
            y.loc[tr_ids], X.loc[ts_ids], y.loc[ts_ids]
        model = deepcopy(model_ref)

        model.fit(X_train, y_train)
        pred = model.predict_proba(X_test)[:, 1]
        val = roc_auc_score(y_test, pred)

        print(round(val, 4))
        hist.append(val)

    hist = np.array(hist)
    if (return_model):
        return model, hist
    return hist

In [30]:
X = r_train[cols]
y = r_train[trg]

classification.set_params(cat_features = [])
m, h = roc_auc_crossval(X, y, classification, return_model = True)
print(h.mean())
imp(m, cols)

0:	learn: 0.6810884	total: 7.54ms	remaining: 2.26s
100:	learn: 0.1947215	total: 464ms	remaining: 913ms
200:	learn: 0.0696726	total: 617ms	remaining: 304ms
299:	learn: 0.0335192	total: 776ms	remaining: 0us
0.5768
0:	learn: 0.6811448	total: 2.47ms	remaining: 738ms
100:	learn: 0.2061148	total: 195ms	remaining: 384ms
200:	learn: 0.0884225	total: 368ms	remaining: 181ms
299:	learn: 0.0459166	total: 536ms	remaining: 0us
0.643
0:	learn: 0.6766776	total: 2.01ms	remaining: 602ms
100:	learn: 0.2069662	total: 165ms	remaining: 325ms
200:	learn: 0.0874690	total: 381ms	remaining: 187ms
299:	learn: 0.0450528	total: 545ms	remaining: 0us
0.6714
0:	learn: 0.6817494	total: 2.7ms	remaining: 806ms
100:	learn: 0.2103240	total: 192ms	remaining: 378ms
200:	learn: 0.0834340	total: 405ms	remaining: 199ms
299:	learn: 0.0430074	total: 565ms	remaining: 0us
0.5978
0:	learn: 0.6803752	total: 1.78ms	remaining: 532ms
100:	learn: 0.1825714	total: 177ms	remaining: 348ms
200:	learn: 0.0658446	total: 333ms	remaining: 164ms

,imp,col
3,25.758181,debt_to_income_known
2,23.648423,credit_history_months
0,18.648192,loan_amount
1,12.171247,monthly_payment
4,11.809420,loan_term_months
5,7.964537,past_due_count_12m


In [60]:
df = pd.merge(r_train, c_train[['client_id', 'inflow_next_30d']], on = 'client_id', how = 'left')
df['pti_ratio'] = df['monthly_payment'] / df['inflow_next_30d']
df['loan_to_inflow_ratio'] = df['loan_amount'] / df['inflow_next_30d']

X = df.drop(columns = ['application_id', 'client_id', trg])
y = df[trg]

m, h = roc_auc_crossval(X, y, classification, return_model = True)
print(h.mean())
imp(m, X.columns)

0:	learn: 0.6753535	total: 17.8ms	remaining: 5.33s
100:	learn: 0.1521651	total: 928ms	remaining: 1.83s
200:	learn: 0.0534260	total: 1.94s	remaining: 954ms
299:	learn: 0.0246822	total: 2.54s	remaining: 0us
0.7045
0:	learn: 0.6692343	total: 7.04ms	remaining: 2.11s
100:	learn: 0.1653508	total: 586ms	remaining: 1.15s
200:	learn: 0.0593266	total: 1.18s	remaining: 580ms
299:	learn: 0.0298165	total: 1.75s	remaining: 0us
0.6454
0:	learn: 0.6718360	total: 6.26ms	remaining: 1.87s
100:	learn: 0.1607182	total: 639ms	remaining: 1.26s
200:	learn: 0.0526079	total: 1.22s	remaining: 599ms
299:	learn: 0.0255525	total: 1.86s	remaining: 0us
0.7376
0:	learn: 0.6742531	total: 5.78ms	remaining: 1.73s
100:	learn: 0.1768315	total: 689ms	remaining: 1.36s
200:	learn: 0.0643346	total: 927ms	remaining: 456ms
299:	learn: 0.0341240	total: 1.14s	remaining: 0us
0.65
0:	learn: 0.6739160	total: 2.32ms	remaining: 694ms
100:	learn: 0.1579030	total: 217ms	remaining: 428ms
200:	learn: 0.0503232	total: 439ms	remaining: 216ms

,imp,col
4,18.341444,credit_history_months
0,14.282528,loan_amount
5,13.323276,debt_to_income_known
7,11.478321,pti_ratio
8,10.818838,loan_to_inflow_ratio
2,9.848511,monthly_payment
1,9.327411,loan_term_months
6,7.190491,inflow_next_30d
3,5.389180,past_due_count_12m


### 3.4 Legal inflow extraction

In [64]:
from sklearn.preprocessing import StandardScaler

model_ref = deepcopy(clients_model)

rX = r_train[cols + ['client_id']]
ry = r_train[trg]
kfold_outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=67)

fold_idx = 1
outer_scores = []

for tr_idx, ts_idx in kfold_outer.split(rX, ry):
    
    train_credit = r_train.loc[tr_idx].copy()
    test_credit = r_train.loc[ts_idx].copy()
    
    test_credit_clients = test_credit['client_id'].unique()
    
    c_train_outer = c_train[~c_train['client_id'].isin(test_credit_clients)].reset_index(drop=True)
    c_test_outer = c_train[c_train['client_id'].isin(test_credit_clients)].reset_index(drop=True)
    
    oof_train_inflow = np.zeros(len(c_train_outer))
    kfold_inner = KFold(n_splits=4, shuffle=True, random_state=67)
    
    for inner_tr, inner_ts in kfold_inner.split(c_train_outer):
        X_in_tr = c_train_outer.loc[inner_tr, clients_cols]
        X_in_ts = c_train_outer.loc[inner_ts, clients_cols]
        
        y_in_tr_scaled = c_train_outer.loc[inner_tr, 'inflow_next_30d'] / 1000
        
        scaler_inner = StandardScaler()
        X_in_tr_scaled = scaler_inner.fit_transform(X_in_tr)
        X_in_ts_scaled = scaler_inner.transform(X_in_ts)
        
        inner_model = deepcopy(model_ref)
        inner_model.fit(X_in_tr_scaled, y_in_tr_scaled)
        
        oof_train_inflow[inner_ts] = inner_model.predict(X_in_ts_scaled) * 1000
        
    dt_train_inflow = pd.DataFrame({
        "client_id": c_train_outer['client_id'],
        "pred_inflow": oof_train_inflow
    })
    
    X_out_tr = c_train_outer[clients_cols]
    X_out_ts = c_test_outer[clients_cols]
    y_out_tr_scaled = c_train_outer['inflow_next_30d'] / 1000
    
    scaler_outer = StandardScaler()
    X_out_tr_scaled = scaler_outer.fit_transform(X_out_tr)
    X_out_ts_scaled = scaler_outer.transform(X_out_ts)
    
    final_inflow_model = deepcopy(model_ref)
    final_inflow_model.fit(X_out_tr_scaled, y_out_tr_scaled)
    
    pred_test_inflow_scaled = final_inflow_model.predict(X_out_ts_scaled)
    dt_test_inflow = pd.DataFrame({
        "client_id": c_test_outer['client_id'],
        "pred_inflow": pred_test_inflow_scaled * 1000
    })
    
    train_df = pd.merge(train_credit, dt_train_inflow, on='client_id', how='left')
    test_df = pd.merge(test_credit, dt_test_inflow, on='client_id', how='left')
    
    actual_test_inflow = c_test_outer.set_index('client_id').loc[test_df['client_id']]['inflow_next_30d']
    print(f"RMSE: {root_mean_squared_error(test_df['pred_inflow'], actual_test_inflow)}")
    
    for df_split in [train_df, test_df]:
        df_split['pred_inflow'] = np.clip(df_split['pred_inflow'], 1.0, None)
        df_split['pti_ratio'] = df_split['monthly_payment'] / df_split['pred_inflow']
        df_split['loan_to_inflow_ratio'] = df_split['loan_amount'] / df_split['pred_inflow']
        
    final_features = cols + ['pred_inflow', 'pti_ratio', 'loan_to_inflow_ratio']
    
    X_train = train_df[final_features]
    y_train = train_df['overdue_next_60d']
    X_test = test_df[final_features]
    y_test = test_df['overdue_next_60d']
    
    risk_model = deepcopy(classification)
    risk_model.set_params(verbose=0)
    risk_model.fit(X_train, y_train)
    
    pred_risk_proba = risk_model.predict_proba(X_test)[:, 1]
    metric = roc_auc_score(y_test, pred_risk_proba)
    print(f"ROC-AUC: {metric}")
    print()
    
    outer_scores.append(metric)
    fold_idx += 1

print(np.mean(outer_scores))

RMSE: 534.1160105157753
ROC-AUC: 0.5400641025641026

RMSE: 486.6320963820775
ROC-AUC: 0.6111111111111112

RMSE: 432.1613671759497
ROC-AUC: 0.6931818181818182

0.6147856772856773
